# AnimationStudio — Cloud-Backend Generation (Phases 1–3)

**No GPU required.** This notebook generates the character reference,
world-environment, and prop libraries through cloud ML APIs (fal.ai,
Replicate, Black Forest Labs). Cloud is the **only realistic path** to the
full 12,472-asset library: the free T4 runs fp8 Flux at ~2 min/image, which
is ~17 days for the full Phase 1–3 scope. No 17 GB model download, no ComfyUI.

Requirements:
- a `FAL_API_KEY` / `REPLICATE_API_KEY` / `BFL_API_KEY` (entered via getpass)
- a GitHub PAT if the repo is private (entered via getpass, same cell)

Steps:
1. Cell 1 — set `REPO_URL` and `CLOUD_PROVIDER`.
2. Cell 3 — paste your cloud key (+ GitHub PAT for private repos).
3. Run Cells 4–6 to generate each library (re-run to resume — already-
   generated assets are skipped with `skip_scored=True`).
4. Cell 7 validates `catalog.db`; Cell 8 syncs to GitHub.


In [ ]:
#@title 1. Settings

import getpass
import os
import subprocess
import sys

# GitHub clone URL for this studio (colab-gpu is the only supported branch).
REPO_URL = "https://github.com/YOUR_ORG/AnimationStudio.git"  #@param {type:"string"}

# Branch: colab-gpu -> fp8 Flux bundle. master is deprecated/unused (N-02).
BRANCH = 'colab-gpu'  #@param ['colab-gpu']

# Cloud ML provider: fal | replicate | bfl
CLOUD_PROVIDER = 'fal'  #@param ['fal', 'replicate', 'bfl']

# Write each generated image into the catalog tree.
PERSIST_IMAGES = True  #@param {type:"boolean"}

# Push every N variant groups to git on the generation checkpoint.
SYNC_EVERY = 1  #@param {type:"integer"}

# In-flight safety: push each image the moment it is generated so a Colab
# termination loses at most the single in-flight image.
SYNC_EVERY_IMAGE = True  #@param {type:"boolean"}

# Cap the run to a slice for a first smoke test (0 = full scope).
LIMIT = 5  #@param {type:"integer"}

# Validate settings/keys only — no API calls, no generation.
DRY_RUN = False  #@param {type:"boolean"}

assert BRANCH == 'colab-gpu', f'Only colab-gpu is supported; got {BRANCH}'
assert CLOUD_PROVIDER in ('fal', 'replicate', 'bfl'), CLOUD_PROVIDER

ENV_KEY_MAP = {
    'fal': 'FAL_API_KEY',
    'replicate': 'REPLICATE_API_KEY',
    'bfl': 'BFL_API_KEY',
}
print(f'Cloud provider: {CLOUD_PROVIDER}')
print(f'Branch: {BRANCH}')
print(f'Limit: {LIMIT or "full scope"}')


In [ ]:
#@title 2. Clone repo + install minimal deps

REPO = '/content/AnimationStudio'
DB = f'{REPO}/catalog.db'
GIT_NAME = 'Colab Studio'
GIT_EMAIL = 'colab@animationstudio.local'

if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1',
                    REPO_URL, REPO], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'requests', 'Pillow', 'pydantic'], check=True)

os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, f'{REPO}/colab')

from src.generation_engine.cloud_backend import CloudAPIBackend  # noqa: E402
from git_sync import auto_sync  # noqa: E402


def _gen_cmd(script: str, extra: list) -> list:
    """Build the full cloud-generation argv for one phase script."""
    cmd = [
        sys.executable, script,
        '--backend', 'cloud',
        '--provider', CLOUD_PROVIDER,
        '--db', DB,
        '--world', f'{REPO}/World',
        '--universe', f'{REPO}/Universe',
        '--assets', f'{REPO}/Assets',
    ]
    if PERSIST_IMAGES:
        cmd += ['--persist-images']
    else:
        cmd += ['--no-persist-images']
    if LIMIT:
        cmd += ['--limit', str(LIMIT)]
    if SYNC_EVERY > 0:
        cmd += ['--sync-every', str(SYNC_EVERY)]
    if SYNC_EVERY_IMAGE:
        cmd += ['--sync-every-image']
    cmd += [
        '--sync-branch', BRANCH,
        '--sync-token', GITHUB_TOKEN,
        '--sync-remote-url', REPO_URL,
        '--sync-git-name', GIT_NAME,
        '--sync-git-email', GIT_EMAIL,
    ]
    return cmd + extra


def _run_gen(script: str, extra: list) -> int:
    cmd = _gen_cmd(script, extra)
    print('$ ' + ' '.join(cmd))
    res = subprocess.run(cmd)
    return res.returncode

print('CloudAPIBackend + git_sync imports OK')


In [ ]:
#@title 3. Enter secrets (runtime-only, never saved to disk)

env_key = ENV_KEY_MAP[CLOUD_PROVIDER]
key = getpass.getpass(f'Enter {env_key}: ')
os.environ[env_key] = key

# Public repos work without a token; private repos need a PAT.
GITHUB_TOKEN = getpass.getpass('Enter GitHub PAT (empty = public repo): ')

if DRY_RUN:
    print('DRY_RUN=True — skipping API validation.')
else:
    backend = CloudAPIBackend(provider=CLOUD_PROVIDER)
    backend.load_model()
    if not backend._ready:
        raise SystemExit(
            f'{CLOUD_PROVIDER} backend not ready — check your API key.'
        )
    print(f'{CLOUD_PROVIDER} backend validated successfully.')


In [ ]:
#@title 4. Phase 1 — character reference library (cloud)

if DRY_RUN:
    print('DRY_RUN=True — skipping Phase 1.')
else:
    rc = _run_gen('scripts/generate_phase1_library.py', extra=[])
    print(f'Phase 1 generation cell finished (rc={rc}).')


In [ ]:
#@title 5. Phase 2 — world library (cloud)

if DRY_RUN:
    print('DRY_RUN=True — skipping Phase 2.')
else:
    rc = _run_gen('scripts/generate_phase2_world.py', extra=[])
    print(f'Phase 2 generation cell finished (rc={rc}).')


In [ ]:
#@title 6. Phase 3 — global asset library (cloud)

if DRY_RUN:
    print('DRY_RUN=True — skipping Phase 3.')
else:
    rc = _run_gen('scripts/generate_phase3_assets.py', extra=[])
    print(f'Phase 3 generation cell finished (rc={rc}).')


In [ ]:
#@title 7. Validate the asset catalog

os.chdir(REPO)
subprocess.run([sys.executable, 'scripts/verify_catalog.py', '--db', DB])
print('Catalog verification finished.')


In [ ]:
#@title 8. Final sync to GitHub

# Generation cells already pushed incrementally (--sync-every-image).
# This is a safety net for any stragglers / the DB state.
if GITHUB_TOKEN.strip():
    os.chdir(REPO)
    from datetime import datetime
    auto_sync(repo=REPO, branch=BRANCH, db_path=DB, token=GITHUB_TOKEN,
              remote_url=REPO_URL,
              git_name=GIT_NAME, git_email=GIT_EMAIL,
              message=f'Cloud backlog {datetime.now():%Y-%m-%d %H:%M}')
else:
    print('No GITHUB_TOKEN — skipped final sync.')


## Next steps

1. Review generated images in the Review UI (`scripts/review_ui.py` or the
   Phase 2/3 notebooks' tunnel cell) and approve assets (`approved` in
   `catalog.db`).
2. After ≥20 approved references per character, run
   `scripts/train_lora.py build-dataset` then the training notebook
   (`AnimationStudio_Colab_Training.ipynb`) for LoRA training.
3. Cloud costs are pay-per-image (≈$0.01–0.03/image on fal/Replicate Flux);
   a full-library run is the only way to reach the 12,472-asset VISION
   pipeline coverage — preview the scope with `LIMIT` first.
